# Example 3: Adding Magnets

This notebook covers how to hollow out hemispheres and add magnet joint cavities so the globe can be assembled magnetically. We will demonstrate two alternatives:
1. Placing magnets with reinforcing bosses (cylindrical towers).
2. Placing magnets inside the natural shell thickness (no bosses).
We will also generate a calibration test piece to tune the tolerances.

## Step 1: Import libraries

In [ ]:
import os
import trimesh
from globe3d import (
    GlobeModel,
    MagnetSettings,
    generate_magnet_test_piece
)

## Step 2: Prepare base spheres

We generate outer sphere vertices and an inner sphere that defines the hollow cavity (sized to 80% of the outer radius).

In [ ]:
model_radius_mm = 40.0
# Initialize GlobeModel
model = GlobeModel.from_fibonacci(n_points=6000, radius=model_radius_mm)
# Generate inner mesh with specified wall thickness (diameter reduction: 2 * (40.0 - 40.0 * 0.75) = 20.0 mm thickness)
model.create_inner_mesh(thickness=20.0)

## Step 3: Alternative A: Hollowing with magnet BOSSES (`add_bosses=True`)

We configure a dictionary of magnet parameters. The boolean engine will automatically union plastic cylinders (bosses) around each magnet void so the magnet doesn't breakthrough into the hollow.

In [ ]:
# Set magnet settings with bosses
model.magnet_settings = MagnetSettings(
    diameter=5.0,
    height=2.0,
    n_magnets=3,
    position=0.0,
    horizontal_tolerance=0.15,
    vertical_tolerance=0.10,
    vertical_offset=0.20,
    min_thickness=1.5,
    add_bosses=True
)

# Generate hemispheres using settings
top_boss, bottom_boss = model.generate_hemispheres(
    hollow=True,
    thickness=20.0,
    engine='manifold'
)
print(f"Top boss mesh watertight: {top_boss.is_watertight}")
print(f"Bottom boss mesh watertight: {bottom_boss.is_watertight}")

## Step 4: Alternative B: Hollowing WITHOUT bosses (`add_bosses=False`)

This checks whether magnets can be placed directly inside the shell without protruding bosses. It searches around the cut surface in 2-degree increments and places the magnets in positions that maximize angular spacing uniformity.

In [ ]:
# Configure magnet settings without bosses
model.magnet_settings = MagnetSettings(
    diameter=5.0,
    height=2.0,
    n_magnets=3,
    min_magnets=2,
    min_angular_spacing=60.0,
    step_degrees=2.0,
    horizontal_tolerance=0.15,
    vertical_tolerance=0.10,
    vertical_offset=0.20,
    min_thickness=1.5,
    add_bosses=False
)

top_noboss, bottom_noboss = model.generate_hemispheres(
    hollow=True,
    thickness=20.0,
    engine='manifold'
)
print(f"Top no-boss mesh watertight: {top_noboss.is_watertight}")
print(f"Bottom no-boss mesh watertight: {bottom_noboss.is_watertight}")

## Step 5: Generate a calibration test piece

Instead of printing a full globe to test the fit of your magnets, you can print a small calibration cylinder. Tune `h_tol` (horizontal) and `v_tol` (vertical) so your magnets slide in snugly.

In [ ]:
output_dir = "../outputs"
os.makedirs(output_dir, exist_ok=True)
test_piece_path = os.path.join(output_dir, "magnet_test_piece.stl")

generate_magnet_test_piece(
    diameter=5.0,
    height=2.0,
    horizontal_tolerance=0.15,
    vertical_tolerance=0.10,
    vertical_offset=0.20,
    min_thickness=1.5,
    output_path=test_piece_path
)
print(f"Saved calibration test piece to: {test_piece_path}")